In [0]:
import pandas as pd
from pyspark.sql.functions import col, lit, when

#Global

In [0]:
tb_moni = 'ia.tb_diamond_mod_monitoramento'

tb_entrada = 'diamond_birads.birads.tb_diamond_mod_birads_entrada'
tb_saida = 'diamond_birads.birads.tb_diamond_mod_birads_saida'
tb_envio = 'diamond_birads.birads.vw_diamond_mod_birads_complemento'

#Criacao da tabela de monitoramento

In [0]:
df_entrada = spark.sql(f"""
    select 
        dt_execucao,
        emp_regional_unidade as nm_regional,
        COUNT(id_exame) AS qtd_entrada,
        emp_nome_unidade as nm_unidade,
        id_unidade
    from {tb_entrada}
    where emp_regional_unidade is not null
    GROUP BY dt_execucao,emp_regional_unidade,emp_nome_unidade,id_unidade

    ORDER BY dt_execucao
""")

df_saida = spark.sql(f"""
    SELECT
        a.dt_execucao,
        emp_regional_unidade as nm_regional,
        COUNT(a.id_exame) AS qtd_saida,
        id_unidade
    FROM {tb_saida} as a
    left join (
        SELECT DISTINCT id_exame,emp_regional_unidade
        FROM {tb_entrada}
    ) as b
        ON a.id_exame = b.id_exame
    where emp_regional_unidade is not null
    GROUP BY a.dt_execucao,emp_regional_unidade,id_unidade
    ORDER BY a.dt_execucao
""")

df_envio = spark.sql(f"""
    select 
        dt_execucao,
        emp_regional_unidade as nm_regional,
        COUNT(id_exame) AS qtd_envio,
        id_unidade
    from {tb_envio}
    where emp_regional_unidade is not null
    GROUP BY dt_execucao,emp_regional_unidade,id_unidade
    ORDER BY dt_execucao
""")

df_relevante = spark.sql(f"""
    select 
        dt_execucao,
        emp_regional_unidade as nm_regional,
        COUNT(id_exame) AS qtd_relevante,
        id_unidade
    from {tb_envio}
    where emp_regional_unidade is not null 
    and vl_proced_birads in (4,5) 
    GROUP BY dt_execucao,emp_regional_unidade,id_unidade
    ORDER BY dt_execucao
""")

In [0]:
df_monitoramento = (
    df_entrada.alias("e")
        .join(df_saida.alias("s"), on=["dt_execucao", "nm_regional", "id_unidade"], how="left")
        .join(df_envio.alias("v"), on=["dt_execucao", "nm_regional", "id_unidade"], how="left")
        .join(df_relevante.alias("r"), on=["dt_execucao", "nm_regional", "id_unidade"], how="left")
        .select(
            "dt_execucao",
            "e.qtd_entrada",
            "s.qtd_saida",
            "v.qtd_envio",
            "r.qtd_relevante",
            "e.nm_unidade",
            "e.nm_regional",
        )
)

In [0]:
df_monitoramento = (
    df_monitoramento
        .withColumn("qtd_entrada", col("qtd_entrada").cast("int"))
        .withColumn("qtd_saida", col("qtd_saida").cast("int"))
        .withColumn("qtd_envio", col("qtd_envio").cast("int"))
        .withColumn("qtd_relevante", col("qtd_relevante").cast("int"))
        .withColumn("nm_regional", col("nm_regional").cast("string"))
        .withColumn("nm_unidade", col("nm_unidade").cast("string"))
        .withColumn("nm_projeto", lit("Mama").cast("string"))
)

In [0]:
df_monitoramento.createOrReplaceTempView("vw_monitoramento_wrk_01")


In [0]:
# %sql
# MERGE INTO ia.tb_diamond_mod_monitoramento t
# USING vw_monitoramento_wrk_01 s
# ON  t.dt_execucao = s.dt_execucao
# AND t.nm_regional = s.nm_regional
# AND t.nm_projeto  = s.nm_projeto

# WHEN MATCHED THEN
#   UPDATE SET
#     t.qtd_relevante = s.qtd_relevante;


In [0]:
%sql
MERGE INTO ia.tb_diamond_mod_monitoramento t
USING vw_monitoramento_wrk_01 s
ON  t.dt_execucao = s.dt_execucao
AND t.nm_regional = s.nm_regional
AND t.nm_unidade = s.nm_unidade
AND t.nm_projeto  = s.nm_projeto

WHEN NOT MATCHED THEN
  INSERT (
    dt_execucao,
    qtd_entrada,
    qtd_saida,
    qtd_envio,
    qtd_relevante,
    nm_regional,
    nm_unidade,
    nm_projeto
  )
  VALUES (
    s.dt_execucao,
    s.qtd_entrada,
    s.qtd_saida,
    s.qtd_envio,
    s.qtd_relevante,
    s.nm_regional,
    s.nm_unidade,
    s.nm_projeto
  )


In [0]:
# Use esse bloco apenas se a tabela de monitoramento estiver vazia(Drop Table)
# df_monitoramento.write.mode("append").saveAsTable(tbl_moni)

In [0]:
dbutils.notebook().exit("Fim da execução!")

In [0]:
%sql
select 
dt_execucao,
qtd_entrada,
qtd_saida,
qtd_envio,
qtd_relevante,
nm_regional,
nm_unidade
from ia.tb_diamond_mod_monitoramento 
where nm_projeto = "Mama" 
--and nm_unidade is not null
--and nm_regional = "RJ" 
--and dt_execucao = "2026-02-05"